# Node-Level Frontend Pipeline For Llama 405B

实现 Llama 405B decoder layer 的最小前端全流程，并在 code gen pass 之后按 node 读取 `marco_op_list`，对每个 node 单独运行硬件仿真。

## 流程
- 加载 `llama-405B` model card 并初始化 `LlamaModelConfig`
- 初始化 `NandConfig`、`InferenceConfig`，支持配置 `TP_SIZE`
- 初始化 `LlamaDecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass` 生成全局 `macro_op_list`
- 遍历每个 node 的 `node.meta['marco_op_list']`
- 对每个非空 node 的 macro op list 单独运行模拟器，输出 node 信息、macro op 信息和模拟时间

In [1]:
import json
import logging
from collections import Counter
from dataclasses import fields, is_dataclass
from pathlib import Path
from pprint import pprint

import torch
from torch.fx import GraphModule

from nandmachine.commands.macro import AllReduceOp, FlashAttnOp, MacroOp, MatMulOp, VectorOp
from nandmachine.config.config import NandConfig
from nandmachine.config.inference_config import DenseParallelConfig, InferenceConfig
from nandmachine.config.model_config import LlamaModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.llama import LlamaDecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state
from nandmachine.simulator.entry_point import run_macro_ops

MODEL_CARD_PATH = Path("model_cards/llama-405B.json")
DEVICE_NAME = "H100_SXM"
COMPILE_MODE = "heuristic-GPU"
TP_SIZE = 8

assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH
assert TP_SIZE > 0, TP_SIZE


def node_summary(graph_module: GraphModule) -> list[tuple[str, str, str]]:
    return [(node.name, node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


def node_info(node: torch.fx.Node) -> dict[str, object]:
    return {
        "node_name": node.name,
        "node_op": node.op,
        "node_target": str(node.target),
    }


def _serialize_value(value: object) -> object:
    if isinstance(value, MacroOp):
        return {
            "id": value.id,
            "type": type(value).__name__,
        }
    if isinstance(value, list):
        return [_serialize_value(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_serialize_value(item) for item in value)
    return value


def macro_info(op: MacroOp) -> dict[str, object]:
    assert is_dataclass(op), type(op)
    payload = {field.name: _serialize_value(getattr(op, field.name)) for field in fields(op)}
    payload["type"] = type(op).__name__
    return payload


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)


In [3]:
model_config = LlamaModelConfig.from_dict(json.loads(MODEL_CARD_PATH.read_text()))
nand_config = NandConfig(
    num_channels=6,
    num_plane=96,
    num_block=16,
    num_pages=16,
    tRead=4000,
    tWrite=4000 * 10,
    tErase=4000 * 100,
    page_size=4,
    sram_threshold=1024 * 80,
)
parallel_config = DenseParallelConfig(
    num_ranks=TP_SIZE,
    tp_size=TP_SIZE,
    dp_size=1,
)
inference_config = InferenceConfig(
    batch_size=128,
    input_sequence_length=8096,
    output_sequence_length=1024,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024 * 256,
    memory_backend="nand",
    parallel_config=parallel_config,
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)
simulator_config = {
    "device_name": DEVICE_NAME,
    "compile_mode": COMPILE_MODE,
}

print(model_config)
print(parallel_config)
print(inference_config)
print(kv_cache_state)
print(simulator_config)


LlamaModelConfig(attention_type='gqa', hidden_size=16384, num_attention_heads=128, num_key_value_heads=8, max_position_embeddings=131072, intermediate_size=53248, hidden_act='silu', head_dim=128, rms_norm_eps=1e-05, attention_bias=False, mlp_bias=False, rope_theta=500000.0, rope_scaling={'factor': 8.0, 'low_freq_factor': 1.0, 'high_freq_factor': 4.0, 'original_max_position_embeddings': 8192, 'rope_type': 'llama3'}, num_hidden_layers=126)
DenseParallelConfig(num_ranks=8, tp_size=8, dp_size=1)
InferenceConfig(batch_size=128, input_sequence_length=8096, output_sequence_length=1024, weight_bits=16, activation_bits=16, kv_cache_bits=16, kv_block_size_bytes=262144, memory_backend='nand', parallel_config=DenseParallelConfig(num_ranks=8, tp_size=8, dp_size=1))
KVCacheState(total_kv_cache_size_per_layer=597688320, kv_block_size_tokens=512, num_kv_blocks=2280, num_nand_pages_per_layer=145920, num_hyper_pages_per_layer=1520)
{'device_name': 'H100_SXM', 'compile_mode': 'heuristic-GPU'}


In [4]:
with torch.device("meta"):
    model = LlamaDecoderLayer(model_config, tp_size=TP_SIZE)
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]
node_macro_op_entries = []

for node in graph_module.graph.nodes:
    if "marco_op_list" not in node.meta:
        continue
    node_macro_op_entries.append((node, node.meta["marco_op_list"]))

print("normalized node count:", len(node_summary(graph_module)))
print("node count with macro op entries:", len(node_macro_op_entries))
print("global macro op count:", len(macro_op_list))
print("global macro op type summary:", dict(macro_summary(macro_op_list)))
print("first 10 vector ops:", [(op.vector_op_type, op.vector_shape) for op in vector_ops[:10]])


2026-04-02 20:44:50,937 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-04-02 20:44:50,937 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=3
2026-04-02 20:44:50,937 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-04-02 20:44:50,938 - INFO - Processed node=self_attn_attn generated_macro_ops=570
2026-04-02 20:44:50,938 - INFO - Processed node=self_attn_o_proj generated_macro_ops=4
2026-04-02 20:44:50,939 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-04-02 20:44:50,939 - INFO - Processed node=mlp_gate_up_proj generated_macro_ops=18
2026-04-02 20:44:50,939 - INFO - Processed node=mlp_act_fn generated_macro_ops=1
2026-04-02 20:44:50,939 - INFO - Processed node=mlp_down_proj generated_macro_ops=10


normalized node count: 13
node count with macro op entries: 9
global macro op count: 608
global macro op type summary: {'VectorOp': 3, 'SramPrefetch': 201, 'MatMulOp': 11, 'SramPrefetchRelease': 201, 'FlashAttnOp': 190, 'AllReduceOp': 2}
first 10 vector ops: [('rms_norm', [128, 16384]), ('rms_norm', [128, 16384]), ('silu_mul', [128, 6656])]


In [5]:
assert macro_op_list
assert node_macro_op_entries
assert any(isinstance(op, MatMulOp) for op in macro_op_list)
assert any(isinstance(op, FlashAttnOp) for op in macro_op_list)
assert any(op.vector_op_type == "rms_norm" for op in vector_ops)
assert any(op.vector_op_type == "silu_mul" for op in vector_ops)
assert all(all(dim > 0 for dim in op.vector_shape) for op in vector_ops)
if TP_SIZE > 1:
    assert any(isinstance(op, AllReduceOp) for op in macro_op_list)

print("llama node-level frontend pipeline completed successfully")


llama node-level frontend pipeline completed successfully


In [6]:
node_sim_results = []

for node, node_macro_ops in node_macro_op_entries:
    current_node_info = node_info(node)
    current_macro_summary = dict(macro_summary(node_macro_ops))

    print("=" * 120)
    print("node info:")
    pprint(current_node_info)
    print("macro op count:", len(node_macro_ops))
    print("macro op type summary:", current_macro_summary)

    serialized_macro_ops = [macro_info(op) for op in node_macro_ops]
    print("macro op details:")
    for macro_op_payload in serialized_macro_ops:
        pprint(macro_op_payload)

    if not node_macro_ops:
        print("simulation skipped because this node produced no macro ops")
        node_sim_results.append(
            {
                **current_node_info,
                "macro_op_count": 0,
                "macro_op_types": [],
                "sim_cycle": None,
                "sim_time_ns": None,
                "macro_ops": serialized_macro_ops,
            }
        )
        continue

    sim_result = run_macro_ops(
        nand_config,
        node_macro_ops,
        **simulator_config,
    )
    assert sim_result.cycle > 0
    assert sim_result.time_ns > 0
    print("sim cycle:", sim_result.cycle)
    print("sim time ns:", sim_result.time_ns)

    node_sim_results.append(
        {
            **current_node_info,
            "macro_op_count": len(node_macro_ops),
            "macro_op_types": [type(op).__name__ for op in node_macro_ops],
            "sim_cycle": sim_result.cycle,
            "sim_time_ns": sim_result.time_ns,
            "macro_ops": serialized_macro_ops,
        }
    )


node info:
{'node_name': 'input_layernorm',
 'node_op': 'call_module',
 'node_target': 'input_layernorm'}
macro op count: 1
macro op type summary: {'VectorOp': 1}
macro op details:
{'id': 1,
 'input_ops': [],
 'type': 'VectorOp',
 'vector_op_type': 'rms_norm',
 'vector_shape': [128, 16384],
 'weight_bits': 16}
sim cycle: 251
sim time ns: 137
node info:
{'node_name': 'self_attn_qkv_proj',
 'node_op': 'call_module',
 'node_target': 'self_attn.qkv_proj'}
macro op count: 3
macro op type summary: {'SramPrefetch': 1, 'MatMulOp': 1, 'SramPrefetchRelease': 1}
macro op details:
{'id': 2, 'input_ops': [], 'num_prefetch_pages': 18432, 'type': 'SramPrefetch'}
{'dim': (128, 16384, 2304),
 'id': 3,
 'input_ops': [{'id': 2, 'type': 'SramPrefetch'}],
 'type': 'MatMulOp',
 'weight_bits': 16}
{'id': 4,
 'input_ops': [{'id': 3, 'type': 'MatMulOp'}],
 'type': 'SramPrefetchRelease'}
sim cycle: 1450962
sim time ns: 792875
node info:
{'node_name': 'self_attn_rotary_emb',
 'node_op': 'call_module',
 'node_tar

In [7]:
node_sim_results


[{'node_name': 'input_layernorm',
  'node_op': 'call_module',
  'node_target': 'input_layernorm',
  'macro_op_count': 1,
  'macro_op_types': ['VectorOp'],
  'sim_cycle': 251,
  'sim_time_ns': 137,
  'macro_ops': [{'id': 1,
    'input_ops': [],
    'vector_op_type': 'rms_norm',
    'vector_shape': [128, 16384],
    'weight_bits': 16,
    'type': 'VectorOp'}]},
 {'node_name': 'self_attn_qkv_proj',
  'node_op': 'call_module',
  'node_target': 'self_attn.qkv_proj',
  'macro_op_count': 3,
  'macro_op_types': ['SramPrefetch', 'MatMulOp', 'SramPrefetchRelease'],
  'sim_cycle': 1450962,
  'sim_time_ns': 792875,
  'macro_ops': [{'id': 2,
    'input_ops': [],
    'num_prefetch_pages': 18432,
    'type': 'SramPrefetch'},
   {'id': 3,
    'input_ops': [{'id': 2, 'type': 'SramPrefetch'}],
    'dim': (128, 16384, 2304),
    'weight_bits': 16,
    'type': 'MatMulOp'},
   {'id': 4,
    'input_ops': [{'id': 3, 'type': 'MatMulOp'}],
    'type': 'SramPrefetchRelease'}]},
 {'node_name': 'self_attn_rotary_

In [8]:
non_empty_node_results = [result for result in node_sim_results if result["macro_op_count"] > 0]

assert non_empty_node_results
assert all(result["sim_cycle"] is not None for result in non_empty_node_results)
assert all(result["sim_time_ns"] is not None for result in non_empty_node_results)

summary_rows = [
    {
        "node_name": result["node_name"],
        "node_op": result["node_op"],
        "node_target": result["node_target"],
        "macro_op_count": result["macro_op_count"],
        "macro_op_types": result["macro_op_types"],
        "sim_cycle": result["sim_cycle"],
        "sim_time_ns": result["sim_time_ns"],
    }
    for result in node_sim_results
]

summary_rows


[{'node_name': 'input_layernorm',
  'node_op': 'call_module',
  'node_target': 'input_layernorm',
  'macro_op_count': 1,
  'macro_op_types': ['VectorOp'],
  'sim_cycle': 251,
  'sim_time_ns': 137},
 {'node_name': 'self_attn_qkv_proj',
  'node_op': 'call_module',
  'node_target': 'self_attn.qkv_proj',
  'macro_op_count': 3,
  'macro_op_types': ['SramPrefetch', 'MatMulOp', 'SramPrefetchRelease'],
  'sim_cycle': 1450962,
  'sim_time_ns': 792875},
 {'node_name': 'self_attn_rotary_emb',
  'node_op': 'call_module',
  'node_target': 'self_attn.rotary_emb',
  'macro_op_count': 0,
  'macro_op_types': [],
  'sim_cycle': None,
  'sim_time_ns': None},
 {'node_name': 'self_attn_attn',
  'node_op': 'call_module',
  'node_target': 'self_attn.attn',
  'macro_op_count': 570,
  'macro_op_types': ['SramPrefetch',
   'FlashAttnOp',
   'SramPrefetchRelease',
   'SramPrefetch',
   'FlashAttnOp',
   'SramPrefetchRelease',
   'SramPrefetch',
   'FlashAttnOp',
   'SramPrefetchRelease',
   'SramPrefetch',
   'F